In [0]:
%pip install esmf-aspect-model-loader==2.3.0 pydeequ==1.5.0 pydantic==2.13.4 oauthlib==3.3.1 annotated-types==0.7.0 jinja2==3.1.2

In [0]:
%run /Workspace/RBS/.bundle/rbs_python_sdk_0.9.13/files/rbs_python_sdk/set_python_path_for_bundles

In [0]:
# Source (raw layer) and target (harmonized layer)
SOURCE_TABLE = "bci_rbs_dev.raw.me_bdo_organization_all"
EMPLOYEE_COUNT_TABLE = "bci_rbs_dev.raw.me_bdo_organization_employee_count"  # OrgId -> EmployeeCount
TARGET_CATALOG = "bci_rbs_dev"
TARGET_SCHEMA = "harmonized"
TARGET_TABLE = "me_bdo_organization"
TARGET_FQN = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

# Aspect model: aspect file + shared files (property definitions live in the shared files)
MODEL_DIR = ("/Workspace/Users/nec9bue@bosch.com/ME_BDO_Reporting/Models/"
             "com.bosch.mobility.mobilityelectronics.businessdigitalorganization.organization/1.0.0")
TTL_FILES = [
    "Organization.ttl",
    "OrganizationalOffice_shared.ttl",
    "CostCentre_shared.ttl",
    "PlanningObject_shared.ttl",
    "Location_shared.ttl",
]

# Raw column -> aspect property column (snake_case, as generated by SammCli)
COLUMN_MAPPING = {
    "ManagerOrgID":          "organizational_office_id",
    "Department":            "organizational_office_name",
    "ParentOrgID":           "has_parent_organizational_office",
    "ManagerCostCenter":     "cost_centre_id",
    "ManagerPlanningObject": "planning_object_id",
    "ManagerLocation":       "location_code",
}

# Business key: must be present, one record per key
REQUIRED_COLUMNS = ["organizational_office_id"]

# Model properties without a source column (filled with NULL until a source is available)
# Employee counts are derived from EMPLOYEE_COUNT_TABLE in cell 06
NULLABLE_OVERRIDES = []

print(f"Source: {SOURCE_TABLE}\nTarget: {TARGET_FQN}")

In [0]:
import os
from common.aspect_model_loader import AspectModelLoader

# Concatenate aspect + shared TTL so all property definitions resolve in one graph
ttl_content = ""
for fname in TTL_FILES:
    path = os.path.join(MODEL_DIR, fname)
    assert os.path.exists(path), f"TTL file not found: {path}"
    with open(path, "r") as f:
        ttl_content += f.read() + "\n"

loader = AspectModelLoader()
samm_graph = loader.get_aspect_model(model_data_string=ttl_content, source="data_string")
samm_graph.load_aspect_model()

aspect = samm_graph.aspect
print(f"Loaded aspect: {aspect.name}")
print(f"URN:           {aspect.urn}")
print(f"Properties:    {[p.name for p in aspect.properties]}")

In [0]:
import tempfile, subprocess, sys, re

# SammCli.to_sql() generates CREATE TABLE (snake_case columns, types, comments, URN property).
# Run in a subprocess to capture its output reliably in notebooks.
with tempfile.NamedTemporaryFile(mode="w", suffix=".ttl", delete=False) as f:
    f.write(ttl_content)
    _ttl_tmp = f.name
try:
    result = subprocess.run(
        [sys.executable, "-c",
         f"from esmf_aspect_meta_model_python.samm_cli import SammCli; SammCli().to_sql('{_ttl_tmp}')"],
        capture_output=True, text=True,
    )
    assert result.returncode == 0, f"SammCli error:\n{result.stderr}"
    generated_sql = result.stdout.strip().rstrip(";")
finally:
    os.unlink(_ttl_tmp)

# Point the DDL at the target table
create_table_sql = re.sub(r"CREATE TABLE IF NOT EXISTS\s+\S+",
                          f"CREATE OR REPLACE TABLE {TARGET_FQN}", generated_sql, count=1)

# Relax NOT NULL for model properties without a source column
for col in NULLABLE_OVERRIDES:
    create_table_sql = re.sub(rf"(\b{col}\s+\w+)\s+NOT NULL", r"\1", create_table_sql)

# Add technical columns (not part of the aspect model) before the closing column bracket
create_table_sql = re.sub(
    r"\n\)\s*\nCOMMENT",
    ",\n  _load_date TIMESTAMP NOT NULL COMMENT 'Load timestamp of the harmonization run.',"
    "\n  _record_source STRING NOT NULL COMMENT 'Source table of the record.'\n)\nCOMMENT",
    create_table_sql, count=1,
)

print(create_table_sql)

In [0]:
from pyspark.sql import functions as F

raw_df = spark.read.table(SOURCE_TABLE)

# Rename to aspect property columns; blank strings -> NULL
mapped_df = raw_df.select([
    F.nullif(F.trim(F.col(src)), F.lit("")).alias(tgt) for src, tgt in COLUMN_MAPPING.items()
])
# Model properties without a source column
for col in NULLABLE_OVERRIDES:
    mapped_df = mapped_df.withColumn(col, F.lit(None).cast("string"))

# Employees per organizational office: join employee counts on OrgId (left join keeps orgs without a count)
emp_df = spark.read.table(EMPLOYEE_COUNT_TABLE).select(
    F.trim(F.col("OrgId")).alias("organizational_office_id"),
    F.col("EmployeeCount").cast("decimal(10,0)").alias("number_of_employees_organizational_office"),
)
mapped_df = mapped_df.join(emp_df, "organizational_office_id", "left")

# Employees per planning object: sum of office employee counts per ManagerPlanningObject
po_emp_df = (
    mapped_df.filter(F.col("planning_object_id").isNotNull())
    .groupBy("planning_object_id")
    .agg(F.sum("number_of_employees_organizational_office").cast("decimal(10,0)")
          .alias("number_of_employees_planning_object"))
)
mapped_df = mapped_df.join(po_emp_df, "planning_object_id", "left")

# Required columns (business key); all other aspect properties are optional
is_valid = F.lit(True)
for c in REQUIRED_COLUMNS:
    is_valid = is_valid & F.col(c).isNotNull()

# Anomalies: rows without a business key (kept for review, not written)
anomalies_df = mapped_df.filter(~is_valid).distinct()

# Harmonized: valid rows, one record per business key
harmonized_df = (
    mapped_df.filter(is_valid)
    .dropDuplicates(REQUIRED_COLUMNS)
    .withColumn("_load_date", F.current_timestamp())
    .withColumn("_record_source", F.lit(SOURCE_TABLE))
)

print(f"Raw rows:          {raw_df.count()}")
print(f"Anomaly rows:      {anomalies_df.count()}")
print(f"Harmonized rows:   {harmonized_df.count()}")
display(anomalies_df)
display(harmonized_df.limit(10))

In [0]:
# Create table from the model-derived DDL (keeps comments, URN property and existing tags)
spark.sql(create_table_sql)

# Overwrite data; columns ordered like the target table (insertInto is positional)
target_cols = spark.table(TARGET_FQN).columns
harmonized_df.select(target_cols).write.insertInto(TARGET_FQN, overwrite=True)

print(f"Written {spark.table(TARGET_FQN).count()} rows to {TARGET_FQN}")
display(spark.sql(f"DESCRIBE TABLE {TARGET_FQN}"))
display(spark.table(TARGET_FQN).limit(5))

In [0]:
from common.aspect_model_client import AspectModelClient
from databricks_client.table_metadata_manager import TableMetadataManager

DRY_RUN = False  # True = only log the generated SQL, do not execute

# Map table columns to aspect model properties
client = AspectModelClient(samm_model=samm_graph, data_frame=spark.table(TARGET_FQN))

manager = TableMetadataManager(
    spark=spark,
    catalog=TARGET_CATALOG,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE,
    log_sql_statements=True,  # show generated COMMENT / SET TAGS statements
)

# Table comment + aspect URN tag, column comments + property URN tags
success, errors = manager.apply_aspect_model_metadata(client, dry_run=DRY_RUN)
print(f"Success: {success}")
for err in errors:
    print(f"[{err.operation}] {err.error}")

# Verify applied tags
display(spark.sql(f"""
    SELECT 'TABLE' AS scope, NULL AS column_name, tag_name, tag_value
    FROM {TARGET_CATALOG}.information_schema.table_tags
    WHERE schema_name = '{TARGET_SCHEMA}' AND table_name = '{TARGET_TABLE}'
    UNION ALL
    SELECT 'COLUMN', column_name, tag_name, tag_value
    FROM {TARGET_CATALOG}.information_schema.column_tags
    WHERE schema_name = '{TARGET_SCHEMA}' AND table_name = '{TARGET_TABLE}'
    ORDER BY scope DESC, column_name
"""))